In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import numpy as np
from typing import Dict

In [ ]:
with open('data/strategyqa/data/train_gpt3_responses_counterfactual.json', 'r') as f: 
    data = json.load(f)

In [ ]:
def load_llama(model_name: str = "meta-llama/Llama-2-7b-chat-hf"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        # torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


@torch.no_grad()
def generate_answer(
    tokenizer,
    model,
    prompt: str,
    max_new_tokens: int = 16,
    temperature: float = 0.0,
) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0.0),
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
    return generated.strip()

In [ ]:
def build_prompt_yesno(x: str, r: str) -> str:
    """
    Simple StrategyQA-style yes/no prompt.
    """
    return (
        "You are a helpful reasoning assistant.\n"
        "You are given a question and a reasoning chain. "
        "Using ONLY the provided reasoning, answer with a single word: 'yes' or 'no'.\n\n"
        f"Question: {x}\n"
        f"Reasoning:\n{r}\n\n"
        "Final answer (yes or no):"
    )


def normalize_yesno(s: str) -> str:
    s = s.strip().lower()
    if s.startswith("yes"):
        return "yes"
    if s.startswith("no"):
        return "no"
    # fallback: majority vote over first token
    if "yes" in s and "no" not in s:
        return "yes"
    if "no" in s and "yes" not in s:
        return "no"
    return s.split()[0] if s else ""


def score_yesno(pred: str, gold: str) -> int:
    """
    Returns 1 if correct, 0 otherwise.
    gold is assumed to be 'yes' or 'no'.
    """
    p = normalize_yesno(pred)
    g = normalize_yesno(gold)
    return int(p == g)

In [ ]:
def compute_cie_cde(
    data,
    tokenizer,
    model,
    max_new_tokens: int = 8,
    temperature: float = 0.0,
) -> Dict[str, float]:
    acc_00 = []
    acc_01 = []
    # acc_10 = []

    for ex in tqdm(data, desc="Evaluating CIE/CDE"):
        # Y00: X0 with R0
        prompt_00 = build_prompt_yesno(ex['question'], ex['predicted_rationale'])
        y00 = generate_answer(tokenizer, model, prompt_00,
                              max_new_tokens=max_new_tokens,
                              temperature=temperature)
        a00 = score_yesno(y00, ex['gold_label'])
        acc_00.append(a00)

        # Y01: X0 with R1 (change mediator R, keep X)
        prompt_01 = build_prompt_yesno(ex['question'], ex['counterfactual'])
        y01 = generate_answer(tokenizer, model, prompt_01,
                              max_new_tokens=max_new_tokens,
                              temperature=temperature)
        a01 = score_yesno(y01, ex['gold_label'])
        acc_01.append(a01)
        
        # Y10: X1 with R0 (change X, keep R)
        # prompt_10 = build_prompt_yesno(ex.x1, ex.r0)
        # y10 = generate_answer(tokenizer, model, prompt_10,
        #                       max_new_tokens=max_new_tokens,
        #                       temperature=temperature)
        # a10 = score_yesno(y10, ex.label)
        # acc_10.append(a10)

    acc_00_arr = np.array(acc_00, dtype=float)
    acc_01_arr = np.array(acc_01, dtype=float)
    # acc_10_arr = np.array(acc_10, dtype=float)

    cie = float(np.mean(acc_00_arr - acc_01_arr))  # Controlled Indirect Effect
    # cde = float(np.mean(acc_00_arr - acc_10_arr))  # Controlled Direct Effect

    results = {
        "mean_acc_Y00": float(np.mean(acc_00_arr)),
        "mean_acc_Y01": float(np.mean(acc_01_arr)),
        # "mean_acc_Y10": float(np.mean(acc_10_arr)),
        "CIE": cie,
        # "CDE": cde,
    }
    return results




In [ ]:
tokenizer, model = load_llama()

In [ ]:
model_names = ["meta-llama/Llama-2-7b-chat", "meta-llama/Llama-2-7b", "mistralai/Mistral-7B-Instruct-v0.3", "mistralai/Mistral-7B-v0.3"]

In [ ]:
results = []
for model_name in model_names[::-1]: 
    tokenizer, model = load_llama(model_name)
    result = compute_cie_cde(
        data,
        tokenizer,
        model=model,
        max_new_tokens=8,
        temperature=0.0,
        )

    print("Results:")
    for k, v in result.items():
        print(f"{k}: {v:.4f}")

    results.append(result)
    del model 
    torch.cuda.empty_cache()


In [ ]:
import pandas as pd 
results_df = pd.DataFrame(results, index=model_names[::-1][:2])

In [21]:
results_df

,mean_acc_Y00,mean_acc_Y01,CIE
mistralai/Mistral-7B-v0.3,0.070995,0.083738,-0.012743
mistralai/Mistral-7B-Instruct-v0.3,0.682039,0.398058,0.283981
